# Impact preprocessing: standardized 2020–2100 pipeline

This notebook is the **cleaner / more defensible** impact preprocessing pipeline.

You only upload two climate CSVs:

1. `state_summer_heat_story_observed_cmip6_2000_2100.csv`
2. `state_monthly_heat_story_observed_cmip6_2000_2100.csv`

Everything else is remote-loaded when possible:

- EPA ICLUS total population projections through 2100.
- Census ACS current 65+ share, clearly labeled as an older-adult exposure **proxy**.
- NOAA/NWS heat-index formula grid as context, not a projection.
- Optional cooling-degree proxy from your monthly climate data.
- Optional hot-dry proxy only if your monthly data includes precipitation.
- Optional crop exposure proxy is context-only by default, not future yield prediction.

## Output principle

Main defensible result:

`added summer 35°C+ days × projected total population`

Optional proxy layers are labeled with `_proxy` in column names.


In [ ]:
# Install packages used for remote GIS data.
!pip -q install geopandas pyogrio fiona shapely pyproj rtree requests beautifulsoup4 openpyxl tqdm


In [ ]:
import os
import re
import json
import math
import zipfile
import warnings
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("/content/cmip6_heat_story_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR)


## 1. Upload the two climate CSV files

Do **not** use `sample_data`. Upload the two files here and the notebook will move them into `/content/cmip6_heat_story_outputs/`.


In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    src = Path("/content") / filename
    dst = OUTPUT_DIR / filename
    if src.exists():
        shutil.move(str(src), str(dst))
    print("Saved:", dst)

print("\nFiles currently in output folder:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


## 2. Helper functions

In [ ]:
STATE_ABBR_TO_NAME = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California','CO':'Colorado','CT':'Connecticut','DE':'Delaware','DC':'District of Columbia','FL':'Florida','GA':'Georgia','HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana','IA':'Iowa','KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine','MD':'Maryland','MA':'Massachusetts','MI':'Michigan','MN':'Minnesota','MS':'Mississippi','MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada','NH':'New Hampshire','NJ':'New Jersey','NM':'New Mexico','NY':'New York','NC':'North Carolina','ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon','PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota','TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia','WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming'
}
STATE_FIPS_TO_NAME = {
    '01':'Alabama','02':'Alaska','04':'Arizona','05':'Arkansas','06':'California','08':'Colorado','09':'Connecticut','10':'Delaware','11':'District of Columbia','12':'Florida','13':'Georgia','15':'Hawaii','16':'Idaho','17':'Illinois','18':'Indiana','19':'Iowa','20':'Kansas','21':'Kentucky','22':'Louisiana','23':'Maine','24':'Maryland','25':'Massachusetts','26':'Michigan','27':'Minnesota','28':'Mississippi','29':'Missouri','30':'Montana','31':'Nebraska','32':'Nevada','33':'New Hampshire','34':'New Jersey','35':'New Mexico','36':'New York','37':'North Carolina','38':'North Dakota','39':'Ohio','40':'Oklahoma','41':'Oregon','42':'Pennsylvania','44':'Rhode Island','45':'South Carolina','46':'South Dakota','47':'Tennessee','48':'Texas','49':'Utah','50':'Vermont','51':'Virginia','53':'Washington','54':'West Virginia','55':'Wisconsin','56':'Wyoming'
}
STATE_NAMES = set(STATE_ABBR_TO_NAME.values()) | {'District of Columbia'}

def normalize_state(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if not s:
        return np.nan
    su = s.upper()
    if su in STATE_ABBR_TO_NAME:
        return STATE_ABBR_TO_NAME[su]
    if re.fullmatch(r"\d{1,2}", s):
        return STATE_FIPS_TO_NAME.get(s.zfill(2), s)
    return s.title()

def find_first_col(cols, patterns, required=True):
    for pat in patterns:
        regex = re.compile(pat, flags=re.I)
        matches = [c for c in cols if regex.search(str(c))]
        if matches:
            return matches[0]
    if required:
        raise ValueError(f"Could not find column matching any of: {patterns}\nAvailable: {list(cols)}")
    return None

def centered_rolling_available(rows, value_col, window=5, group_cols=None):
    # Centered rolling average with shortened endpoint windows.
    if group_cols is None:
        group_cols = []
    rows = rows.sort_values(group_cols + ['year']).copy()
    out_parts = []
    half = window // 2
    for _, g in rows.groupby(group_cols, dropna=False):
        g = g.sort_values('year').copy().reset_index(drop=True)
        values = g[value_col].astype(float).values
        rolled = []
        win_n = []
        for i in range(len(g)):
            lo = max(0, i - half)
            hi = min(len(g), i + half + 1)
            w = values[lo:hi]
            rolled.append(np.nanmean(w) if np.isfinite(w).any() else np.nan)
            win_n.append(int(np.isfinite(w).sum()))
        g[f'{value_col}_5yr'] = rolled
        g['rolling_window_years'] = win_n
        out_parts.append(g)
    return pd.concat(out_parts, ignore_index=True)

def safe_to_numeric(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '', regex=False), errors='coerce')

print("Helpers ready.")


## 3. Load climate files and create aligned 5-year hot-day series

This is the core climate hazard result.

The notebook finds the existing summer 35°C+ day change column, applies a centered 5-year rolling average with endpoint windows using available years, and aligns each scenario to observed 2020 by subtracting the scenario's 2020 rolling source offset.


In [ ]:
summer_path = OUTPUT_DIR / "state_summer_heat_story_observed_cmip6_2000_2100.csv"
monthly_path = OUTPUT_DIR / "state_monthly_heat_story_observed_cmip6_2000_2100.csv"

assert summer_path.exists(), f"Missing {summer_path.name}. Upload it first."
assert monthly_path.exists(), f"Missing {monthly_path.name}. Upload it first."

summer = pd.read_csv(summer_path)
monthly = pd.read_csv(monthly_path)

summer.columns = [str(c).strip() for c in summer.columns]
monthly.columns = [str(c).strip() for c in monthly.columns]

print("Summer rows/cols:", summer.shape)
print("Monthly rows/cols:", monthly.shape)
print("Potential hot-day columns:")
print([c for c in summer.columns if re.search(r"35|hot|summer", c, flags=re.I)])


In [ ]:
state_col = find_first_col(summer.columns, [r"^state$", r"state_name", r"NAME"])
year_col = find_first_col(summer.columns, [r"^year$"])
scenario_col = find_first_col(summer.columns, [r"^scenario$", r"ssp"])

hot_candidates = [
    c for c in summer.columns
    if re.search(r"35", c, flags=re.I)
    and re.search(r"summer|hot", c, flags=re.I)
    and re.search(r"change|delta|added|increase", c, flags=re.I)
]
if not hot_candidates:
    hot_candidates = [c for c in summer.columns if re.search(r"35", c, flags=re.I) and re.search(r"hot", c, flags=re.I)]
if not hot_candidates:
    raise ValueError("Could not find a summer 35°C+ days column. Please check the summer CSV columns.")

hot_col = hot_candidates[0]
print("Using hot-day column:", hot_col)

hot = summer.copy()
hot = hot.rename(columns={state_col:'state', year_col:'year', scenario_col:'scenario'})
hot['state'] = hot['state'].map(normalize_state)
hot['year'] = pd.to_numeric(hot['year'], errors='coerce').astype('Int64')
hot['scenario'] = hot['scenario'].astype(str).str.lower().str.strip()
hot['hot_days_raw_change'] = safe_to_numeric(hot[hot_col])

hot = hot[hot['year'].between(2020, 2100)].copy()
hot = hot[hot['scenario'].isin(['ssp126','ssp245','ssp585','low','medium','high','observed'])].copy()
hot['scenario'] = hot['scenario'].replace({'low':'ssp126', 'medium':'ssp245', 'high':'ssp585'})
hot_proj = hot[hot['scenario'].isin(['ssp126','ssp245','ssp585'])].copy()

hot_state = (
    hot_proj.groupby(['state','scenario','year'], as_index=False)['hot_days_raw_change']
    .mean()
    .dropna(subset=['state','year','scenario'])
)

hot_roll = centered_rolling_available(
    hot_state,
    value_col='hot_days_raw_change',
    window=5,
    group_cols=['state','scenario']
)

offset_2020 = (
    hot_roll[hot_roll['year'] == 2020][['state','scenario','hot_days_raw_change_5yr']]
    .rename(columns={'hot_days_raw_change_5yr':'cmip6_2020_hotday_source_offset_5yr'})
)
hot_roll = hot_roll.merge(offset_2020, on=['state','scenario'], how='left')
hot_roll['added_summer_35c_days_aligned_5yr'] = (
    hot_roll['hot_days_raw_change_5yr'] - hot_roll['cmip6_2020_hotday_source_offset_5yr']
)
hot_roll['added_summer_35c_days_aligned_5yr_clipped'] = hot_roll['added_summer_35c_days_aligned_5yr'].clip(lower=0)

hot_out = hot_roll[[
    'state','scenario','year','hot_days_raw_change','hot_days_raw_change_5yr',
    'cmip6_2020_hotday_source_offset_5yr',
    'added_summer_35c_days_aligned_5yr','added_summer_35c_days_aligned_5yr_clipped',
    'rolling_window_years'
]].copy()

hot_out_path = OUTPUT_DIR / 'state_hotdays_aligned_rolling_2020_2100.csv'
hot_out.to_csv(hot_out_path, index=False)
hot_2100_path = OUTPUT_DIR / 'state_hotdays_aligned_rolling_2100.csv'
hot_out[hot_out['year'] == 2100].to_csv(hot_2100_path, index=False)

print("Saved:", hot_out_path)
print("Saved:", hot_2100_path)
display(hot_out.head())


## 4. Remote-load EPA ICLUS total population projections

This section computes the main exposure proxy:

`population_exposure_days = added summer 35°C+ days × projected total population`

This is the main impact extension because both components run through 2100.


In [ ]:
ICLUS_POP_URL = "https://dmap-prod-oms-edc.s3.amazonaws.com/ORD/NCEA/ICLUS_v2.1.1/population/ICLUS_v2_1_1_population.zip"

iclus_zip = OUTPUT_DIR / "ICLUS_v2_1_1_population.zip"
iclus_extract = OUTPUT_DIR / "iclus_population"

def download_file(url, out_path, chunk_size=1024*1024):
    if out_path.exists() and out_path.stat().st_size > 0:
        print("Already downloaded:", out_path)
        return out_path
    print("Downloading:", url)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(out_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return out_path

try:
    download_file(ICLUS_POP_URL, iclus_zip)
    iclus_extract.mkdir(exist_ok=True)
    with zipfile.ZipFile(iclus_zip, 'r') as z:
        z.extractall(iclus_extract)
    print("Extracted to:", iclus_extract)
    for p in list(iclus_extract.rglob('*'))[:30]:
        print(p)
except Exception as e:
    print("EPA ICLUS remote load failed.")
    print("Error:", repr(e))
    print("You can still continue with climate-only outputs; population exposure will be skipped.")


In [ ]:
import geopandas as gpd
import pyogrio

pop_state_annual = None

try:
    gdb_candidates = list(iclus_extract.rglob('*.gdb'))
    print("GDB candidates:", gdb_candidates)
    if not gdb_candidates:
        raise FileNotFoundError("No .gdb folder found after extraction.")
    gdb_path = str(gdb_candidates[0])
    layers = pyogrio.list_layers(gdb_path)
    print("Layers:")
    print(layers)
    layer_names = [x[0] for x in layers]
    layer_name = None
    for name in layer_names:
        if re.search(r"pop|population|county", name, flags=re.I):
            layer_name = name
            break
    if layer_name is None:
        layer_name = layer_names[0]
    print("Reading layer:", layer_name)
    pop_gdf = gpd.read_file(gdb_path, layer=layer_name)
    pop_df = pd.DataFrame(pop_gdf.drop(columns='geometry', errors='ignore'))
    print("ICLUS columns:", pop_df.columns.tolist())
    display(pop_df.head())
except Exception as e:
    pop_df = None
    print("Could not read ICLUS geodatabase.")
    print("Error:", repr(e))


In [ ]:
def extract_state_from_iclus(df):
    cols = df.columns
    for c in cols:
        if re.fullmatch(r"state|state_name|stname|name_state", str(c), flags=re.I):
            return df[c].map(normalize_state)
    for c in cols:
        if re.fullmatch(r"statefp|state_fips|stfips|st|STATEFP", str(c), flags=re.I):
            return df[c].astype(str).str.extract(r"(\d{1,2})")[0].str.zfill(2).map(STATE_FIPS_TO_NAME)
    for c in cols:
        if re.search(r"geoid|fips|countyfp|cnty", str(c), flags=re.I):
            vals = df[c].astype(str).str.extract(r"(\d{5})")[0]
            if vals.notna().sum() > 0:
                return vals.str[:2].map(STATE_FIPS_TO_NAME)
    raise ValueError("Could not infer state from ICLUS columns.")

def iclus_wide_to_long(df):
    df = df.copy()
    df['state'] = extract_state_from_iclus(df)
    year_cols = []
    for c in df.columns:
        s = str(c)
        years = re.findall(r"20[2-9]0|2100|20[2-9]5", s)
        if years:
            sample = safe_to_numeric(df[c].dropna().astype(str).head(20))
            if sample.notna().sum() > 0:
                year_cols.append(c)
    if not year_cols:
        raise ValueError("No year population columns found in ICLUS table.")
    print("Detected ICLUS population year columns:", year_cols[:20], "... total", len(year_cols))
    long_parts = []
    for c in year_cols:
        s = str(c).lower()
        year_match = re.search(r"(20[2-9]0|20[2-9]5|2100)", s)
        if not year_match:
            continue
        year = int(year_match.group(1))
        if re.search(r"ssp\s*5|ssp5|_5_|rcp85|85", s):
            pop_scenario = 'ssp5'
        elif re.search(r"ssp\s*2|ssp2|_2_|rcp45|45", s):
            pop_scenario = 'ssp2'
        else:
            pop_scenario = 'unknown'
        tmp = df[['state']].copy()
        tmp['year'] = year
        tmp['population_scenario'] = pop_scenario
        tmp['projected_total_population'] = safe_to_numeric(df[c])
        long_parts.append(tmp)
    long = pd.concat(long_parts, ignore_index=True)
    long = long.dropna(subset=['state','year','projected_total_population'])
    if set(long['population_scenario'].unique()) == {'unknown'}:
        long2 = long.copy(); long2['population_scenario'] = 'ssp2'
        long5 = long.copy(); long5['population_scenario'] = 'ssp5'
        long = pd.concat([long2, long5], ignore_index=True)
    state = (
        long.groupby(['state','population_scenario','year'], as_index=False)['projected_total_population']
        .sum()
    )
    return state

def interpolate_population_annual(pop_state):
    out = []
    for (state, scen), g in pop_state.groupby(['state','population_scenario']):
        g = g.sort_values('year')
        years = pd.DataFrame({'year': np.arange(2020, 2101)})
        tmp = years.merge(g, on='year', how='left')
        tmp['state'] = state
        tmp['population_scenario'] = scen
        tmp['projected_total_population'] = tmp['projected_total_population'].interpolate().ffill().bfill()
        out.append(tmp)
    return pd.concat(out, ignore_index=True)

if pop_df is not None:
    try:
        pop_state = iclus_wide_to_long(pop_df)
        pop_state_annual = interpolate_population_annual(pop_state)
        pop_path = OUTPUT_DIR / 'state_population_projection_annual_2020_2100.csv'
        pop_state_annual.to_csv(pop_path, index=False)
        print("Saved:", pop_path)
        display(pop_state_annual.head())
        display(pop_state_annual.groupby('population_scenario')['year'].agg(['min','max','nunique']))
    except Exception as e:
        pop_state_annual = None
        print("Could not standardize ICLUS population data.")
        print("Error:", repr(e))
else:
    print("Skipping population exposure because ICLUS data was not loaded.")


## 5. Merge hot days with total population exposure

Scenario pairing used here:

- `ssp585` climate → `ssp5` population
- `ssp245` climate → `ssp2` population
- `ssp126` climate → `ssp2` population as a practical available projection pairing

The output is an exposure proxy, not health outcome prediction.


In [ ]:
if pop_state_annual is not None:
    climate_to_pop = {'ssp126':'ssp2', 'ssp245':'ssp2', 'ssp585':'ssp5'}
    exp = hot_out.copy()
    exp['population_scenario'] = exp['scenario'].map(climate_to_pop)
    exp = exp.merge(
        pop_state_annual,
        on=['state','population_scenario','year'],
        how='left'
    )
    exp['population_exposure_days_proxy'] = (
        exp['added_summer_35c_days_aligned_5yr_clipped'] * exp['projected_total_population']
    )
    exp['population_exposure_days_proxy_millions'] = exp['population_exposure_days_proxy'] / 1_000_000
    exp['impact_method'] = 'added_summer_35c_days_aligned_5yr_x_projected_total_population'
    exp_path = OUTPUT_DIR / 'state_hotdays_population_exposure_2020_2100.csv'
    exp.to_csv(exp_path, index=False)
    exp_2100_path = OUTPUT_DIR / 'state_hotdays_population_exposure_2100.csv'
    exp[exp['year'] == 2100].to_csv(exp_2100_path, index=False)
    print("Saved:", exp_path)
    print("Saved:", exp_2100_path)
    display(exp[exp['year'].eq(2100)].sort_values('population_exposure_days_proxy', ascending=False).head(10))
else:
    exp = None
    print("Population exposure output skipped.")


## 6. Optional: current 65+ share proxy

This is **not** a 2100 age projection.

It uses current ACS 65+ share and applies it to projected total population. The output columns include `_proxy` to avoid misinterpretation.


In [ ]:
def load_acs_65plus_share():
    male_65_vars = ['B01001_020E','B01001_021E','B01001_022E','B01001_023E','B01001_024E','B01001_025E']
    female_65_vars = ['B01001_044E','B01001_045E','B01001_046E','B01001_047E','B01001_048E','B01001_049E']
    vars_ = ['NAME','B01001_001E'] + male_65_vars + female_65_vars
    url = 'https://api.census.gov/data/2024/acs/acs5?get=' + ','.join(vars_) + '&for=state:*'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.rename(columns={'NAME':'state','B01001_001E':'acs_total_population_current'})
    for c in ['acs_total_population_current'] + male_65_vars + female_65_vars:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['acs_population_65plus_current'] = df[male_65_vars + female_65_vars].sum(axis=1)
    df['share_65plus_current'] = df['acs_population_65plus_current'] / df['acs_total_population_current']
    df['state'] = df['state'].map(normalize_state)
    df = df[df['state'].isin(STATE_NAMES)].copy()
    return df[['state','share_65plus_current','acs_population_65plus_current','acs_total_population_current']]

try:
    acs65 = load_acs_65plus_share()
    acs65_path = OUTPUT_DIR / 'state_65plus_share_current_acs.csv'
    acs65.to_csv(acs65_path, index=False)
    print("Saved:", acs65_path)
    display(acs65.head())
except Exception as e:
    acs65 = None
    print("ACS 65+ share remote load failed; older-adult proxy will be skipped.")
    print("Error:", repr(e))

if exp is not None and acs65 is not None:
    exp65 = exp.merge(acs65, on='state', how='left')
    exp65['population_65plus_proxy'] = exp65['projected_total_population'] * exp65['share_65plus_current']
    exp65['older_adult_exposure_days_proxy'] = (
        exp65['added_summer_35c_days_aligned_5yr_clipped'] * exp65['population_65plus_proxy']
    )
    exp65['older_adult_exposure_days_proxy_millions'] = exp65['older_adult_exposure_days_proxy'] / 1_000_000
    exp65['older_adult_proxy_method'] = 'current_acs_65plus_share_applied_to_projected_total_population'
    exp65_path = OUTPUT_DIR / 'state_hotdays_population_65plus_proxy_2020_2100.csv'
    exp65_2100_path = OUTPUT_DIR / 'state_hotdays_population_65plus_proxy_2100.csv'
    exp65.to_csv(exp65_path, index=False)
    exp65[exp65['year'] == 2100].to_csv(exp65_2100_path, index=False)
    print("Saved:", exp65_path)
    print("Saved:", exp65_2100_path)
    display(exp65[exp65['year'].eq(2100)].sort_values('older_adult_exposure_days_proxy', ascending=False).head(10))
else:
    exp65 = None
    print("Older-adult exposure proxy skipped.")


## 7. Heat Index context grid

This is context only. It is **not** a projected 2100 result because the project pipeline does not include future humidity.


In [ ]:
def heat_index_fahrenheit(T, RH):
    HI = (-42.379 + 2.04901523*T + 10.14333127*RH - 0.22475541*T*RH
          - 0.00683783*T*T - 0.05481717*RH*RH + 0.00122874*T*T*RH
          + 0.00085282*T*RH*RH - 0.00000199*T*T*RH*RH)
    return HI

def c_to_f(c): return c * 9/5 + 32

def f_to_c(f): return (f - 32) * 5/9

heat_rows = []
for temp_c in np.arange(30, 42.1, 1):
    for rh in [20,30,40,50,60,70,80,90]:
        hi_f = heat_index_fahrenheit(c_to_f(temp_c), rh)
        heat_rows.append({
            'air_temp_c': temp_c,
            'relative_humidity_percent': rh,
            'heat_index_c_context': f_to_c(hi_f),
            'context_only': True,
            'method': 'NOAA_NWS_heat_index_formula_context_grid_not_projection'
        })
heat_index_grid = pd.DataFrame(heat_rows)
heat_index_path = OUTPUT_DIR / 'heat_index_context_grid.csv'
heat_index_grid.to_csv(heat_index_path, index=False)
print("Saved:", heat_index_path)
display(heat_index_grid.head())


## 8. Optional cooling-degree proxy from monthly temperature

This is a proxy for cooling pressure, not electricity demand.


In [ ]:
monthly_cols = monthly.columns.tolist()
state_col_m = find_first_col(monthly_cols, [r"^state$", r"state_name", r"NAME"])
year_col_m = find_first_col(monthly_cols, [r"^year$"])
month_col_m = find_first_col(monthly_cols, [r"^month$"])
scenario_col_m = find_first_col(monthly_cols, [r"^scenario$", r"ssp"])
temp_col_m = find_first_col(monthly_cols, [r"monthly_tas_c", r"tas_c", r"tavg", r"temperature"], required=False)

if temp_col_m is not None:
    m = monthly.copy().rename(columns={state_col_m:'state', year_col_m:'year', month_col_m:'month', scenario_col_m:'scenario'})
    m['state'] = m['state'].map(normalize_state)
    m['year'] = pd.to_numeric(m['year'], errors='coerce')
    m['month'] = pd.to_numeric(m['month'], errors='coerce')
    m['scenario'] = m['scenario'].astype(str).str.lower().str.strip().replace({'low':'ssp126','medium':'ssp245','high':'ssp585'})
    m['monthly_tas_c_for_cooling_proxy'] = safe_to_numeric(m[temp_col_m])
    m = m[m['scenario'].isin(['ssp126','ssp245','ssp585']) & m['year'].between(2020,2100)].copy()
    days_in_month = {1:31,2:28.25,3:31,4:30,5:31,6:30,7:31,8:31,9:30,10:31,11:30,12:31}
    base_c = 18.3
    m['days_in_month'] = m['month'].map(days_in_month)
    m['cooling_degree_days_proxy_monthly'] = np.maximum(m['monthly_tas_c_for_cooling_proxy'] - base_c, 0) * m['days_in_month']
    cdd = (
        m.groupby(['state','scenario','year'], as_index=False)['cooling_degree_days_proxy_monthly']
        .sum()
        .rename(columns={'cooling_degree_days_proxy_monthly':'cooling_degree_days_proxy_annual'})
    )
    cdd_path = OUTPUT_DIR / 'state_cooling_degree_proxy_2020_2100.csv'
    cdd.to_csv(cdd_path, index=False)
    cdd[cdd['year'].eq(2100)].to_csv(OUTPUT_DIR / 'state_cooling_degree_proxy_2100.csv', index=False)
    print("Saved:", cdd_path)
    display(cdd.head())
else:
    print("No monthly temperature column found; cooling-degree proxy skipped.")


## 9. Optional hot-dry compound proxy

This only runs if your monthly file contains precipitation. If no precipitation exists, the notebook exports a context note instead of inventing data.


In [ ]:
precip_col = find_first_col(monthly_cols, [r"precip", r"prcp", r"ppt", r"rain"], required=False)

if precip_col is not None:
    print("Using precipitation column:", precip_col)
    m2 = monthly.copy().rename(columns={state_col_m:'state', year_col_m:'year', month_col_m:'month', scenario_col_m:'scenario'})
    m2['state'] = m2['state'].map(normalize_state)
    m2['year'] = pd.to_numeric(m2['year'], errors='coerce')
    m2['month'] = pd.to_numeric(m2['month'], errors='coerce')
    m2['scenario'] = m2['scenario'].astype(str).str.lower().str.strip().replace({'low':'ssp126','medium':'ssp245','high':'ssp585'})
    m2['precip_value'] = safe_to_numeric(m2[precip_col])
    summer_months = [6,7,8]
    sp = (
        m2[m2['month'].isin(summer_months) & m2['scenario'].isin(['ssp126','ssp245','ssp585'])]
        .groupby(['state','scenario','year'], as_index=False)['precip_value']
        .sum()
        .rename(columns={'precip_value':'summer_precip_value'})
    )
    base = sp[sp['year'].eq(2020)][['state','scenario','summer_precip_value']].rename(columns={'summer_precip_value':'summer_precip_2020'})
    sp = sp.merge(base, on=['state','scenario'], how='left')
    sp['summer_precip_change_from_2020'] = sp['summer_precip_value'] - sp['summer_precip_2020']
    sp['dryness_increase_proxy'] = (-sp['summer_precip_change_from_2020']).clip(lower=0)
    hd = hot_out[['state','scenario','year','added_summer_35c_days_aligned_5yr_clipped']]
    hdry = sp.merge(hd, on=['state','scenario','year'], how='left')
    hdry['hot_dry_compound_proxy'] = hdry['added_summer_35c_days_aligned_5yr_clipped'] * hdry['dryness_increase_proxy']
    hdry['hot_dry_proxy_method'] = 'added_summer_35c_days_aligned_5yr_x_positive_summer_precip_decrease_from_2020'
    hdry_path = OUTPUT_DIR / 'state_hot_dry_compound_proxy_2020_2100.csv'
    hdry.to_csv(hdry_path, index=False)
    hdry[hdry['year'].eq(2100)].to_csv(OUTPUT_DIR / 'state_hot_dry_compound_proxy_2100.csv', index=False)
    print("Saved:", hdry_path)
else:
    context = pd.DataFrame([{
        'context_layer':'hot_dry_compound_risk',
        'status':'not_computed',
        'reason':'monthly climate file did not include precipitation',
        'website_use':'Use as knowledge/context only unless precipitation or drought data is added.'
    }])
    context.to_csv(OUTPUT_DIR / 'hot_dry_context_only.csv', index=False)
    print("No precipitation column found. Saved context-only note.")


## 10. Crop exposure context

This notebook does not predict future crop yield. It creates a context-only note unless you separately decide to add crop acreage data later.


In [ ]:
crop_context = pd.DataFrame([{
    'context_layer':'crop_heat_exposure',
    'status':'context_only_not_computed',
    'reason':'future crop yield is not predicted in this standardized pipeline',
    'recommended_wording':'Crop exposure can be discussed as a downstream context of hot-dry risk; do not claim future crop yield prediction unless adding a crop model or validated crop exposure dataset.'
}])
crop_context.to_csv(OUTPUT_DIR / 'crop_context_only.csv', index=False)
print("Saved crop_context_only.csv")


## 11. Build a compact 2100 impact context table

This table is designed for the website Impact page.


In [ ]:
base_2100 = hot_out[(hot_out['scenario'].eq('ssp585')) & (hot_out['year'].eq(2100))].copy()
base_2100 = base_2100[['state','scenario','year','added_summer_35c_days_aligned_5yr','added_summer_35c_days_aligned_5yr_clipped']]
base_2100 = base_2100.rename(columns={'added_summer_35c_days_aligned_5yr_clipped':'added_summer_35c_days_2100_aligned_5yr'})

impact = base_2100.copy()
if exp is not None:
    cols = ['state','scenario','year','projected_total_population','population_exposure_days_proxy','population_exposure_days_proxy_millions']
    impact = impact.merge(exp[exp['year'].eq(2100)][cols], on=['state','scenario','year'], how='left')
if exp65 is not None:
    cols = ['state','scenario','year','share_65plus_current','population_65plus_proxy','older_adult_exposure_days_proxy','older_adult_exposure_days_proxy_millions']
    impact = impact.merge(exp65[exp65['year'].eq(2100)][cols], on=['state','scenario','year'], how='left')

impact['data_note'] = 'Main exposure uses projected total population. 65+ columns are current-share proxy if present. Values are exposure proxies, not health outcomes.'
impact_path = OUTPUT_DIR / 'state_impact_context_2100.csv'
impact.to_csv(impact_path, index=False)
print("Saved:", impact_path)
display(impact.sort_values('added_summer_35c_days_2100_aligned_5yr', ascending=False).head(10))


## 12. Zip outputs for download

In [ ]:
zip_path = shutil.make_archive(
    base_name=str(OUTPUT_DIR / 'impact_outputs_standardized_2020_2100'),
    format='zip',
    root_dir=str(OUTPUT_DIR),
)
print('Created zip:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('Download manually from:', zip_path)
    print(e)
